<a href="https://colab.research.google.com/github/PGhimire-309/XAI/blob/main/CFEs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install dice-ml
import shap
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import dice_ml
from dice_ml import Dice

**CunterFactuals(CFEs)**


---
What would we need to change for the model to make a different prediction?
*"Smallest change to the input feature that flips the model prediction from orginal to desired class."*

---


"More or less a Optimization Problem"

---


Works on 4 Principles


1.   **Sparsity**: Change as few feature as possible. changinf a whole set of input features would not make sense to end users
2.   **Actionability**: Immutable features can change . In calofornia housing set changing something like latitude doesnot make sense .


1.   **Plausibility**: Realistic data point , cant have again on same datset avgbedrooms -2.
2.   **Causality**: Change must be physically possivle without violating causal laws.





**TYPES**

---


Optimization Based -> Alibi

Genetic Algorithms ->DiCE

Nearest Neighbour ->NICE

Rule Based -> DiCE

In [ ]:


# 1. Load data and train the model
X, y = shap.datasets.california()


y_series = pd.Series(y, name='target')

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y_series)

# 2. Prepare DiCE
data_and_target = pd.concat([X, y_series], axis=1)

d = dice_ml.Data(dataframe=data_and_target,
                 continuous_features=X.columns.tolist(),
                 outcome_name='target')

m = dice_ml.Model(model=model, backend="sklearn", model_type='regressor')
exp = Dice(d, m, method="random")

# 3. Define the instance to explain
query_instance = X.iloc[[0]]




In [ ]:
# 4. Generate Counterfactuals
dice_exp = exp.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_range=[3.0, 5.0],
    features_to_vary=['MedInc', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'HouseAge']
)


100%|██████████| 1/1 [00:00<00:00,  2.93it/s]


In [ ]:

dice_exp.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 4.0)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.02381,322.0,2.555556,37.880001,-122.230003,4.0



Diverse Counterfactual set (new outcome: [3.0, 5.0])


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,-,-,-,-,31974.1,-,-,-,4.453591346740723
1,-,-,-,-,1978.2,-,-,-,4.453591346740723
2,-,8.4,-,-,12794.8,-,-,-,4.274521350860596


This is a fantastic output because it perfectly illustrates exactly how counterfactual algorithms fail in the real world if you don't strictly control them.

Counterfactual 1: It changed Population from 322 to 31,974.

Counterfactual 2: It changed Population to 1,978.

Counterfactual 3: It changed HouseAge from 41 to 8.4 and Population to 12,794.

This is the Causality principle violation. The algorithm is a literal mathematical optimizer. It doesn't know what a "house" is. It only knows that if it changes the number in the Population column, the Random Forest model outputs a higher number.

How to Fix It =Actionability Constraints

In [ ]:
# 1. We ONLY allow features a homeowner can change
actionable_features = ['AveRooms', 'AveBedrms', 'AveOccup']

dice_exp_realistic = exp.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_range=[3.1,4.2],
    features_to_vary=actionable_features
)

dice_exp_realistic.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  3.36it/s]

Query instance (original outcome : 4.0)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.02381,322.0,2.555556,37.880001,-122.230003,4.0



Diverse Counterfactual set (new outcome: [3.1, 4.2])


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,-,-,47.9,-,-,305.0,-,-,4.180771350860596
1,-,-,-,-,-,1034.8,-,-,4.100531101226807
2,-,-,77.1,-,-,174.4,-,-,4.180771350860596


In [ ]:
# 1. We ONLY allow features a homeowner can change
actionable_features = ['AveRooms', 'AveBedrms', 'AveOccup','MedInc']

# 2. Generate strictly constrained Counterfactuals
dice_exp_realistic = exp.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_range=[3.1,4.2], # Force it to find a way to increase the value
    features_to_vary=actionable_features
)

# 3. Visualize
dice_exp_realistic.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  2.06it/s]

Query instance (original outcome : 4.0)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.02381,322.0,2.555556,37.880001,-122.230003,4.0



Diverse Counterfactual set (new outcome: [3.1, 4.2])


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,-,-,-,-,-,560.1,-,-,4.158720970153809
1,-,-,9.6,-,-,444.0,-,-,4.180771350860596
2,7.4765,-,-,-,-,-,-,-,3.8693501949310303


If you want to increase the house value, just make it a 77-room mansion where an average of 1,034 people live per household

Or, if you prefer, just squeeze 560 people into the current house

this is completely absurd and physically impossible.

Only one relsitic answer MedInc = 7.4765 | target = 3.86
This is the only realistic counterfactual in the entire list.

In [ ]:
# Define the realistic bounds for the features
permitted_ranges = {
    'AveRooms': [1, 15],
    'AveBedrms': [1, 5],
    'AveOccup': [1, 10],
    'MedInc': [0.5, 15]
}

dice_exp_realistic = exp.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_range=[4.3, 4.6],
    features_to_vary=['MedInc', 'AveRooms', 'AveBedrms', 'AveOccup'],
    permitted_range=permitted_ranges
)

dice_exp_realistic.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  3.03it/s]

Query instance (original outcome : 4.0)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.02381,322.0,2.555556,37.880001,-122.230003,4.0



Diverse Counterfactual set (new outcome: [4.3, 4.6])


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,-,-,-,-,-,1.1,-,-,4.305861473083496
1,-,-,-,4.8,-,3.8,-,-,4.35710334777832
2,-,-,6.7,-,-,-,-,-,4.381220817565918


Lets analyze  these CFS again

# CF 1 (AveOccup = 1.1)

**Change:** AveOccup drops from 2.55 to 1.1.

**Result:** Predicted value goes to 4.30.

**Interpretation:** If the average number of people per household in this neighborhood drops to 1.1 (essentially single-person households), the model predicts the house is worth more.

**Realism:** Very high. You could theoretically buy the house, kick out the roommates, and have it occupied by one person. The model has learned that lower occupancy correlates with higher property values.

# CF 2 (AveBedrms = 4.8)

**Change:** AveBedrms jumps from 1.02 to 4.8, and AveOccup rises to 3.8.

**Result:** Predicted value goes to 4.35.

**Interpretation:** If you renovate the house to have nearly 5 bedrooms on average, and have 3.8 people living there, the value goes up.

**Realism:** High. Adding bedrooms is a classic way to increase home value. (Note: 4.8 is exactly the upper limit we set in our permitted_range for AveBedrms! The algorithm pushed it right to the ceiling we allowed).


# CF 3 (AveRooms = 6.7):

**Change:** AveRooms drops slightly from 6.98 to 6.7.

**Result:** Predicted value goes to 4.38.

**Interpretation:** "If you actually make the house slightly smaller (fewer rooms), the value goes up."

**Realism:** This seems completely illogical, right? Fewer rooms = more money?

**Why did CF 3 suggest making the house smaller?**

This is a golden example of how Tree-Based Models (like Random Forests) work.

Decision trees don't draw straight lines they draw boxes. The Random Forest likely learned a very specific, non-linear ruleIn . this specific neighborhood, houses with exactly 6.7 to 6.8 rooms are high-density luxury condos, whereas houses with 6.98 rooms are older single-family homes

The algorithm doesn't understand "rooms." It just knows that moving the number from 6.98 to 6.7 crosses a mathematical boundary in the tree that happens to output a higher price.

This is a spurious correlation. It is mathematically true for the model, but causally absurd in the real world.
